# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/mini-gpt-lab")
RUNS_ROOT = DRIVE_ROOT / "runs"
VOCAB_ROOT = DRIVE_ROOT / "vocabs"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
VOCAB_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)
print("VOCAB_ROOT:", VOCAB_ROOT)

In [ ]:
from pathlib import Path
import sys
import time
import json
import importlib
import torch

SRC_DIR = Path("src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import bpe
import dataset
import model as model_module
import train as train_module

importlib.reload(bpe)
importlib.reload(dataset)
importlib.reload(model_module)
importlib.reload(train_module)

BPETokenizer = bpe.BPETokenizer
create_dataloader = dataset.create_dataloader
GPTModel = model_module.GPTModel
train_model = train_module.train_model
load_checkpoint = train_module.load_checkpoint


# ------------------------------------------------------------
# Drive 저장 경로
# ------------------------------------------------------------
if "VOCAB_ROOT" not in globals() or "RUNS_ROOT" not in globals():
    DRIVE_ROOT = Path("/content/drive/MyDrive/mini-gpt-lab")
    RUNS_ROOT = DRIVE_ROOT / "runs"
    VOCAB_ROOT = DRIVE_ROOT / "vocabs"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
VOCAB_ROOT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 기본 모델 1개 설정
# ------------------------------------------------------------
RUN_MODE = "basic"

BASE = {
    "corpus_chars": 1_500_000,
    "val_chars": 50_000,
    "vocab_size": 2_000,
    "context_length": 64,
    "batch_size": 8,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "num_epochs": 40,
    "eval_freq": 100,
    "eval_iter": 10,
    "ckpt_freq": 100,
}

START_CONTEXT = "이 영화는"
SEED = 123

RUN_CONFIGS = [
    {
        "run_name": (
            f"{RUN_MODE}_base_"
            f"vocab{BASE['vocab_size']}_"
            f"ctx{BASE['context_length']}_"
            f"emb{BASE['emb_dim']}_"
            f"layers{BASE['n_layers']}_"
            f"epochs{BASE['num_epochs']}"
        ),
        "changed_param": "base",
        "changed_value": "base",
        "setting": dict(BASE),
    }
]

print("num runs:", len(RUN_CONFIGS))
for item in RUN_CONFIGS:
    print(item["run_name"])


# ------------------------------------------------------------
# 데이터 읽기
# ------------------------------------------------------------
train_text_path = Path("data/nsmc_lm_train.txt")
val_text_path = Path("data/nsmc_lm_val.txt")

if not train_text_path.exists() or not val_text_path.exists():
    raise FileNotFoundError("먼저 데이터 준비 셀 또는 python download_data.py를 실행하세요.")

raw_train_corpus = train_text_path.read_text(encoding="utf-8")
raw_val_corpus = val_text_path.read_text(encoding="utf-8")

train_corpus = raw_train_corpus[:BASE["corpus_chars"]]
val_corpus = raw_val_corpus[:BASE["val_chars"]]

print("train chars:", len(train_corpus))
print("val chars:", len(val_corpus))

assert train_corpus != val_corpus
assert len(train_corpus) > 0
assert len(val_corpus) > 0


# ------------------------------------------------------------
# BPE tokenizer 학습 또는 로드
# ------------------------------------------------------------
vocab_path = VOCAB_ROOT / (
    f"nsmc_bpe_{RUN_MODE}_"
    f"vocab{BASE['vocab_size']}_"
    f"chars{BASE['corpus_chars']}.json"
)

tokenizer = BPETokenizer(vocab_size=BASE["vocab_size"])

start = time.time()

if vocab_path.exists():
    tokenizer.load(vocab_path)
    print("loaded tokenizer:", vocab_path)
else:
    tokenizer.train(train_corpus)
    tokenizer.save(vocab_path)
    print("trained tokenizer:", vocab_path)

print("tokenizer time:", round(time.time() - start, 2), "sec")

sample = "이 영화는 정말 좋았다."
assert tokenizer.decode(tokenizer.encode(sample, add_bos_eos=True), skip_special=True) == sample


# ------------------------------------------------------------
# 학습 실행
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def write_json(path, data):
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


for run_item in RUN_CONFIGS:
    run_name = run_item["run_name"]
    setting = run_item["setting"]

    run_dir = RUNS_ROOT / run_name
    done_path = run_dir / "DONE.json"
    failed_path = run_dir / "FAILED.json"
    resume_path = run_dir / "checkpoints" / "last.pt"

    if done_path.exists():
        print("skip done:", run_name)
        continue

    run_dir.mkdir(parents=True, exist_ok=True)

    if failed_path.exists():
        failed_path.unlink()

    print("=" * 80)
    print("run:", run_name)
    print("=" * 80)

    try:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        train_ids = tokenizer.encode(train_corpus)
        val_ids = tokenizer.encode(val_corpus)

        print("train tokens:", len(train_ids))
        print("val tokens:", len(val_ids))

        train_loader = create_dataloader(
            token_ids=train_ids,
            context_length=setting["context_length"],
            batch_size=setting["batch_size"],
            stride=setting["context_length"],
            shuffle=True,
            drop_last=True,
            num_workers=0,
        )

        val_loader = create_dataloader(
            token_ids=val_ids,
            context_length=setting["context_length"],
            batch_size=setting["batch_size"],
            stride=setting["context_length"],
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )

        print("train batches:", len(train_loader))
        print("val batches:", len(val_loader))

        model_config = {
            "vocab_size": setting["vocab_size"],
            "context_length": setting["context_length"],
            "emb_dim": setting["emb_dim"],
            "n_heads": setting["n_heads"],
            "n_layers": setting["n_layers"],
            "drop_rate": setting["drop_rate"],
            "qkv_bias": False,
        }

        model = GPTModel(model_config).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=setting["learning_rate"],
            weight_decay=setting["weight_decay"],
        )

        start_epoch = 0
        global_step = 0

        if resume_path.exists():
            start_epoch, global_step = load_checkpoint(
                model=model,
                optimizer=optimizer,
                path=str(resume_path),
                device=device,
            )
            print("resumed:", resume_path)
            print("start_epoch:", start_epoch)
            print("global_step:", global_step)

        num_params = sum(p.numel() for p in model.parameters())
        print("model params:", f"{num_params:,}")

        extra_config = {
            "mode": RUN_MODE,
            "seed": SEED,
            "vocab_path": str(vocab_path),
            "changed_param": run_item["changed_param"],
            "changed_value": run_item["changed_value"],
            "train_chars": len(train_corpus),
            "val_chars": len(val_corpus),
            "train_tokens": len(train_ids),
            "val_tokens": len(val_ids),
            "num_params": num_params,
            **setting,
        }

        train_losses, val_losses, track_steps = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            num_epochs=setting["num_epochs"],
            eval_freq=setting["eval_freq"],
            eval_iter=setting["eval_iter"],
            start_context=START_CONTEXT,
            tokenizer=tokenizer,
            ckpt_freq=setting["ckpt_freq"],
            start_epoch=start_epoch,
            global_step=global_step,
            run_dir=run_dir,
            run_name=run_name,
            config=extra_config,
        )

        write_json(done_path, {
            "run_name": run_name,
            "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "changed_param": run_item["changed_param"],
            "changed_value": run_item["changed_value"],
            "final_step": track_steps[-1] if track_steps else global_step,
            "final_train_loss": train_losses[-1] if train_losses else None,
            "final_val_loss": val_losses[-1] if val_losses else None,
            "run_dir": str(run_dir),
            "vocab_path": str(vocab_path),
        })

        print("done:", run_name)

    except Exception as e:
        write_json(failed_path, {
            "run_name": run_name,
            "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "error_type": type(e).__name__,
            "error": str(e),
        })
        print("failed:", run_name, type(e).__name__, e)
        raise

In [ ]:
from pathlib import Path
import sys
import time
import json
import importlib
import torch

SRC_DIR = Path("src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import bpe
import dataset
import model as model_module
import train as train_module

importlib.reload(bpe)
importlib.reload(dataset)
importlib.reload(model_module)
importlib.reload(train_module)

BPETokenizer = bpe.BPETokenizer
create_dataloader = dataset.create_dataloader
GPTModel = model_module.GPTModel
train_model = train_module.train_model
load_checkpoint = train_module.load_checkpoint


# ------------------------------------------------------------
# 기본 실험 설정
# ------------------------------------------------------------
SWEEP_MODE = "light"

BASE = {
    "corpus_chars": 500_000,
    "val_chars": 50_000,
    "vocab_size": 2_000,
    "context_length": 64,
    "batch_size": 8,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "num_epochs": 20,
    "eval_freq": 100,
    "eval_iter": 10,
    "ckpt_freq": 100,
}

# 변인 통제:
# 한 번에 하나의 하이퍼파라미터만 바꿉니다.
# 각 항목은 small / base / large 3개입니다.
SWEEP_VALUES = {
    "learning_rate": [1e-4, BASE["learning_rate"], 5e-4],
    "batch_size": [4, BASE["batch_size"], 16],
    "drop_rate": [0.0, BASE["drop_rate"], 0.5],
    "context_length": [32, BASE["context_length"], 128],
    "emb_dim": [64, BASE["emb_dim"], 192],
    "n_heads": [2, BASE["n_heads"], 8],
    "n_layers": [2, BASE["n_layers"], 6],
}

START_CONTEXT = "이 영화는"
SEED = 123


# ------------------------------------------------------------
# 데이터 읽기
# ------------------------------------------------------------
train_text_path = Path("data/nsmc_lm_train.txt")
val_text_path = Path("data/nsmc_lm_val.txt")

if not train_text_path.exists() or not val_text_path.exists():
    raise FileNotFoundError("먼저 데이터 준비 셀 또는 python download_data.py를 실행하세요.")

raw_train_corpus = train_text_path.read_text(encoding="utf-8")
raw_val_corpus = val_text_path.read_text(encoding="utf-8")

train_corpus = raw_train_corpus[:BASE["corpus_chars"]]
val_corpus = raw_val_corpus[:BASE["val_chars"]]

print("train chars:", len(train_corpus))
print("val chars:", len(val_corpus))


# ------------------------------------------------------------
# tokenizer는 모든 모델이 공통으로 씁니다.
# vocab까지 바꾸면 tokenizer 자체가 달라져서 모델 비교가 지저분해집니다.
# ------------------------------------------------------------
vocab_path = VOCAB_ROOT / (
    f"nsmc_bpe_{SWEEP_MODE}_"
    f"vocab{BASE['vocab_size']}_"
    f"chars{BASE['corpus_chars']}.json"
)

tokenizer = BPETokenizer(vocab_size=BASE["vocab_size"])

start = time.time()

if vocab_path.exists():
    tokenizer.load(vocab_path)
    print("loaded tokenizer:", vocab_path)
else:
    tokenizer.train(train_corpus)
    tokenizer.save(vocab_path)
    print("trained tokenizer:", vocab_path)

print("tokenizer time:", round(time.time() - start, 2), "sec")

sample = "이 영화는 정말 좋았다."
assert tokenizer.decode(tokenizer.encode(sample, add_bos_eos=True), skip_special=True) == sample


# ------------------------------------------------------------
# 한 변인씩 바꾸는 run 목록 생성
# ------------------------------------------------------------
def make_run_configs(base, sweep_values):
    runs = []

    for param_name, values in sweep_values.items():
        for value in values:
            cfg = dict(base)
            cfg[param_name] = value

            if cfg["emb_dim"] % cfg["n_heads"] != 0:
                print("skip invalid:", param_name, value, "emb_dim % n_heads != 0")
                continue

            label = "base" if value == base[param_name] else str(value).replace(".", "p")
            run_name = f"{SWEEP_MODE}_{param_name}_{label}"

            runs.append({
                "run_name": run_name,
                "changed_param": param_name,
                "changed_value": value,
                "setting": cfg,
            })

    return runs


RUN_CONFIGS = make_run_configs(BASE, SWEEP_VALUES)

print("num runs:", len(RUN_CONFIGS))
for item in RUN_CONFIGS:
    print(item["run_name"])


# ------------------------------------------------------------
# 학습 실행
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def write_json(path, data):
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


for run_item in RUN_CONFIGS:
    run_name = run_item["run_name"]
    setting = run_item["setting"]
    run_dir = RUNS_ROOT / run_name
    done_path = run_dir / "DONE.json"
    failed_path = run_dir / "FAILED.json"

    if done_path.exists():
        print("skip done:", run_name)
        continue

    run_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("run:", run_name)
    print("changed:", run_item["changed_param"], "=", run_item["changed_value"])
    print("=" * 80)

    try:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        train_ids = tokenizer.encode(train_corpus)
        val_ids = tokenizer.encode(val_corpus)

        train_loader = create_dataloader(
            token_ids=train_ids,
            context_length=setting["context_length"],
            batch_size=setting["batch_size"],
            stride=setting["context_length"],
            shuffle=True,
            drop_last=True,
            num_workers=0,
        )

        val_loader = create_dataloader(
            token_ids=val_ids,
            context_length=setting["context_length"],
            batch_size=setting["batch_size"],
            stride=setting["context_length"],
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )

        model_config = {
            "vocab_size": setting["vocab_size"],
            "context_length": setting["context_length"],
            "emb_dim": setting["emb_dim"],
            "n_heads": setting["n_heads"],
            "n_layers": setting["n_layers"],
            "drop_rate": setting["drop_rate"],
            "qkv_bias": False,
        }

        model = GPTModel(model_config).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=setting["learning_rate"],
            weight_decay=setting["weight_decay"],
        )

        start_epoch = 0
        global_step = 0

        resume_path = run_dir / "checkpoints" / "last.pt"
        if resume_path.exists():
            start_epoch, global_step = load_checkpoint(
                model=model,
                optimizer=optimizer,
                path=str(resume_path),
                device=device,
            )
            print("resumed:", resume_path)
            print("start_epoch:", start_epoch)
            print("global_step:", global_step)

        num_params = sum(p.numel() for p in model.parameters())

        extra_config = {
            "mode": SWEEP_MODE,
            "seed": SEED,
            "vocab_path": str(vocab_path),
            "changed_param": run_item["changed_param"],
            "changed_value": run_item["changed_value"],
            "train_chars": len(train_corpus),
            "val_chars": len(val_corpus),
            "train_tokens": len(train_ids),
            "val_tokens": len(val_ids),
            "num_params": num_params,
            **setting,
        }

        train_losses, val_losses, track_steps = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            num_epochs=setting["num_epochs"],
            eval_freq=setting["eval_freq"],
            eval_iter=setting["eval_iter"],
            start_context=START_CONTEXT,
            tokenizer=tokenizer,
            ckpt_freq=setting["ckpt_freq"],
            start_epoch=start_epoch,
            global_step=global_step,
            run_dir=run_dir,
            run_name=run_name,
            config=extra_config,
        )

        write_json(done_path, {
            "run_name": run_name,
            "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "changed_param": run_item["changed_param"],
            "changed_value": run_item["changed_value"],
            "final_step": track_steps[-1] if track_steps else global_step,
            "final_train_loss": train_losses[-1] if train_losses else None,
            "final_val_loss": val_losses[-1] if val_losses else None,
        })

        print("done:", run_name)

    except Exception as e:
        write_json(failed_path, {
            "run_name": run_name,
            "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "error_type": type(e).__name__,
            "error": str(e),
        })
        print("failed:", run_name, type(e).__name__, e)
        print("다음 run으로 넘어갑니다.")
        continue

In [ ]:
from pathlib import Path
import json
import re
import textwrap
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm


# =========================
# 0. Colab 한글 폰트 설정
# =========================

try:
    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

    if not Path(font_path).exists():
        !apt-get -qq update
        !apt-get -qq install -y fonts-nanum

    fm.fontManager.addfont(font_path)

    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False

    mpl.rcParams["font.family"] = "NanumGothic"
    mpl.rcParams["axes.unicode_minus"] = False

except Exception as e:
    print("한글 폰트 설정 실패:", e)


# =========================
# 1. 경로 설정
# =========================

RUNS_ROOT = Path("/content/drive/MyDrive/mini-gpt-lab/runs")
BASE_RUN_NAME = "base"


# =========================
# 2. 파일 읽기
# =========================

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path):
    rows = []

    if not path.exists():
        return rows

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows


# =========================
# 3. run 정보 읽기
# =========================

def get_best_metric(metrics):
    valid_rows = [
        row for row in metrics
        if row.get("val_loss") is not None
    ]

    if not valid_rows:
        return None

    return min(valid_rows, key=lambda row: row["val_loss"])


def get_last_metric_per_epoch(metrics):
    by_epoch = {}

    for row in metrics:
        epoch = row.get("epoch")

        if epoch is None:
            continue

        by_epoch[epoch] = row

    epochs = sorted(by_epoch.keys())
    rows = [by_epoch[epoch] for epoch in epochs]

    return epochs, rows


def get_sample_near_best_step(run):
    samples = run["samples"]

    if not samples:
        return None

    best_metric = run["best_metric"]

    if best_metric is None:
        return samples[-1]

    best_step = best_metric.get("step", 0)

    return min(
        samples,
        key=lambda sample: abs(sample.get("step", 0) - best_step),
    )


def parse_run(run_dir):
    config_path = run_dir / "config.json"
    metrics_path = run_dir / "metrics.jsonl"
    samples_path = run_dir / "samples.jsonl"
    summary_path = run_dir / "summary.json"

    if not config_path.exists() or not metrics_path.exists():
        return None

    config = read_json(config_path)
    metrics = read_jsonl(metrics_path)
    samples = read_jsonl(samples_path)
    summary = read_json(summary_path) if summary_path.exists() else {}

    if not metrics:
        return None

    return {
        "name": run_dir.name,
        "path": run_dir,
        "config": config,
        "metrics": metrics,
        "samples": samples,
        "summary": summary,
        "best_metric": get_best_metric(metrics),
    }


runs = []

for run_dir in sorted(RUNS_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue

    run = parse_run(run_dir)

    if run is not None:
        runs.append(run)

print("loaded runs:", len(runs))

for run in runs:
    best = run["best_metric"]
    best_text = f"best val loss: {best['val_loss']:.4f}" if best else "no val loss"
    print(f"- {run['name']} | {best_text}")


# =========================
# 4. config 값 가져오기
# =========================

def get_config_value(run, key):
    config = run["config"]

    if key in config:
        return config[key]

    candidate_sections = [
        "model_config",
        "train_config",
        "data_config",
        "tokenizer_config",
        "extra",
        "data",
        "model",
        "training",
    ]

    for section_name in candidate_sections:
        section = config.get(section_name, {})

        if isinstance(section, dict) and key in section:
            return section[key]

    return None


def find_run_by_name(run_name):
    for run in runs:
        if run["name"] == run_name:
            return run

    return None


base_run = find_run_by_name(BASE_RUN_NAME)

if base_run is None:
    raise ValueError("runs/base 폴더를 찾지 못했습니다.")


# =========================
# 5. 비교 그룹 만들기
# =========================

GROUPS = {
    "vocab_size": [
        "vocab_size_300",
        "base",
        "vocab_size_10000",
    ],
    "n_layers": [
        "n_layers_1",
        "base",
        "n_layers_10",
    ],
    "emb_dim": [
        "emb_dim_4",
        "base",
        "emb_dim_256",
    ],
    "drop_rate": [
        "drop_rate_0p0",
        "base",
        "drop_rate_0p5",
    ],
    "context_length": [
        "context_length_8",
        "base",
        "context_length_512",
    ],
}


PARAM_KEYS = {
    "vocab_size": "vocab_size",
    "n_layers": "n_layers",
    "emb_dim": "emb_dim",
    "drop_rate": "drop_rate",
    "context_length": "context_length",
}


PARAM_LABELS = {
    "vocab_size": "vocab size",
    "n_layers": "number of layers",
    "emb_dim": "embedding dimension",
    "drop_rate": "dropout rate",
    "context_length": "context length",
}


grouped_runs = {}

for group_name, run_names in GROUPS.items():
    group = []

    for run_name in run_names:
        run = find_run_by_name(run_name)

        if run is None:
            print(f"[skip] {run_name} 폴더를 찾지 못했습니다.")
            continue

        group.append(run)

    grouped_runs[group_name] = group


def get_max_epoch(run):
    epochs = []

    for row in run["metrics"]:
        epoch = row.get("epoch")

        if epoch is not None:
            epochs.append(epoch)

    if not epochs:
        return 0

    return max(epochs)


def is_epoch100_run(run):
    name = run["name"].lower()

    if re.search(r"(epoch|epochs)[_-]?100\b", name):
        return True

    max_epoch = get_max_epoch(run)

    if max_epoch >= 100:
        return True

    return False


epoch100_runs = [
    run for run in runs
    if is_epoch100_run(run)
]


# =========================
# 6. 그래프 스타일
# =========================

ROLE_COLORS = {
    "small": "#1f77b4",
    "base": "#ff7f0e",
    "large": "#2ca02c",
    "other": "#9467bd",
}


def get_param_value(run, group_name):
    param_key = PARAM_KEYS[group_name]

    return get_config_value(run, param_key)


def get_role(run, group_name):
    if run["name"] == "base":
        return "base"

    value = get_param_value(run, group_name)
    base_value = get_param_value(base_run, group_name)

    if value is None or base_value is None:
        return "other"

    if value < base_value:
        return "small"

    if value > base_value:
        return "large"

    return "base"


def clean_sample_text(text, max_chars=360):
    if text is None:
        return "(sample 없음)"

    text = str(text)
    text = text.replace("\n", " / ")
    text = " ".join(text.split())

    if len(text) > max_chars:
        text = text[:max_chars] + "..."

    return text


def get_generated_text_from_sample(sample):
    if sample is None:
        return "(samples.jsonl 없음)"

    possible_keys = [
        "generated_text",
        "sample_text",
        "text",
        "output",
        "generation",
    ]

    for key in possible_keys:
        if key in sample and sample[key] is not None:
            return sample[key]

    return str(sample)


# =========================
# 7. loss 그래프
# =========================

def plot_epoch_loss(ax, group_name, group):
    for run in group:
        role = get_role(run, group_name)
        color = ROLE_COLORS.get(role, ROLE_COLORS["other"])
        value = get_param_value(run, group_name)

        epochs, rows = get_last_metric_per_epoch(run["metrics"])

        train_losses = [row.get("train_loss") for row in rows]
        val_losses = [row.get("val_loss") for row in rows]

        label_prefix = f"{run['name']} ({PARAM_LABELS[group_name]}={value})"

        ax.plot(
            epochs,
            train_losses,
            linestyle="--",
            color=color,
            alpha=0.65,
            linewidth=1.8,
            label=f"{label_prefix} train",
        )

        ax.plot(
            epochs,
            val_losses,
            linestyle="-",
            color=color,
            linewidth=2.6,
            label=f"{label_prefix} val",
        )

        best = run["best_metric"]

        if best is not None:
            ax.scatter(
                best["epoch"],
                best["val_loss"],
                color=color,
                s=80,
                zorder=5,
            )

    ax.set_xlabel("epoch", fontsize=13)
    ax.set_ylabel("loss", fontsize=13)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=9)


# =========================
# 8. 생성 문장 패널
# =========================

def draw_sample_panel(ax, group_name, group):
    ax.axis("off")

    y = 0.98

    for run in group:
        role = get_role(run, group_name)
        color = ROLE_COLORS.get(role, ROLE_COLORS["other"])

        value = get_param_value(run, group_name)
        param_key = PARAM_KEYS[group_name]

        sample = get_sample_near_best_step(run)
        generated_text = get_generated_text_from_sample(sample)

        header = f"{param_key} = {value}"

        ax.text(
            0.00,
            y,
            header,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            color=color,
            va="top",
        )

        y -= 0.075

        text = clean_sample_text(generated_text, max_chars=360)
        wrapped = textwrap.fill(text, width=44)

        ax.text(
            0.02,
            y,
            wrapped,
            transform=ax.transAxes,
            fontsize=18,
            color="black",
            va="top",
            linespacing=1.35,
        )

        line_count = len(wrapped.splitlines())
        y -= 0.085 * line_count + 0.11

        if y < 0.05:
            break


# =========================
# 9. 하이퍼파라미터별 그래프 출력
# =========================

for group_name, group in grouped_runs.items():
    if not group:
        continue

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(24, 8),
        gridspec_kw={"width_ratios": [1.2, 1.05]},
    )

    plot_epoch_loss(axes[0], group_name, group)
    draw_sample_panel(axes[1], group_name, group)

    plt.tight_layout()
    plt.show()


# =========================
# 10. epoch100 run이 있으면 따로 출력
# =========================

if epoch100_runs:
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(24, 8),
        gridspec_kw={"width_ratios": [1.2, 1.05]},
    )

    loss_ax = axes[0]

    for index, run in enumerate(epoch100_runs):
        color = plt.cm.tab10(index % 10)

        epochs, rows = get_last_metric_per_epoch(run["metrics"])

        train_losses = [row.get("train_loss") for row in rows]
        val_losses = [row.get("val_loss") for row in rows]

        loss_ax.plot(
            epochs,
            train_losses,
            linestyle="--",
            color=color,
            alpha=0.65,
            linewidth=1.8,
            label=f"{run['name']} train",
        )

        loss_ax.plot(
            epochs,
            val_losses,
            linestyle="-",
            color=color,
            linewidth=2.6,
            label=f"{run['name']} val",
        )

        best = run["best_metric"]

        if best is not None:
            loss_ax.scatter(
                best["epoch"],
                best["val_loss"],
                color=color,
                s=80,
                zorder=5,
            )

    loss_ax.set_title("epoch 100 run - epoch 기준", fontsize=16)
    loss_ax.set_xlabel("epoch", fontsize=13)
    loss_ax.set_ylabel("loss", fontsize=13)
    loss_ax.grid(True, alpha=0.25)
    loss_ax.legend(fontsize=9)

    sample_ax = axes[1]
    sample_ax.axis("off")
    sample_ax.set_title("epoch 100 생성 문장", fontsize=18, fontweight="bold", pad=12)

    y = 0.98

    for index, run in enumerate(epoch100_runs):
        color = plt.cm.tab10(index % 10)

        sample = get_sample_near_best_step(run)
        generated_text = get_generated_text_from_sample(sample)

        sample_ax.text(
            0.00,
            y,
            run["name"],
            transform=sample_ax.transAxes,
            fontsize=15,
            fontweight="bold",
            color=color,
            va="top",
        )

        y -= 0.075

        text = clean_sample_text(generated_text, max_chars=360)
        wrapped = textwrap.fill(text, width=44)

        sample_ax.text(
            0.02,
            y,
            wrapped,
            transform=sample_ax.transAxes,
            fontsize=18,
            color="black",
            va="top",
            linespacing=1.35,
        )

        line_count = len(wrapped.splitlines())
        y -= 0.085 * line_count + 0.11

    fig.suptitle(
        "epoch 100 추가 학습 결과",
        fontsize=20,
        fontweight="bold",
    )

    plt.tight_layout()
    plt.show()


# =========================
# 11. basic_base run도 같은 형식으로 출력
# =========================

TARGET_RUN_NAME = "basic_base_vocab2000_ctx64_emb128_layers4_epochs40"

target_run = find_run_by_name(TARGET_RUN_NAME)

if target_run is not None:
    def plot_single_run_epoch_loss(ax, run):
        epochs, rows = get_last_metric_per_epoch(run["metrics"])

        train_losses = [row.get("train_loss") for row in rows]
        val_losses = [row.get("val_loss") for row in rows]

        color = "#d62728"

        ax.plot(
            epochs,
            train_losses,
            linestyle="--",
            color=color,
            alpha=0.65,
            linewidth=1.8,
            label=f"{run['name']} train",
        )

        ax.plot(
            epochs,
            val_losses,
            linestyle="-",
            color=color,
            linewidth=2.6,
            label=f"{run['name']} val",
        )

        best = run["best_metric"]

        if best is not None:
            ax.scatter(
                best["epoch"],
                best["val_loss"],
                color=color,
                s=80,
                zorder=5,
                label=f"best val {best['val_loss']:.4f}",
            )

        ax.set_title("basic_base 비교 - epoch 기준", fontsize=16)
        ax.set_xlabel("epoch", fontsize=13)
        ax.set_ylabel("loss", fontsize=13)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=9)


    def draw_single_run_sample_panel(ax, run):
        ax.axis("off")

        ax.set_title(
            "basic_base 생성 문장",
            fontsize=18,
            fontweight="bold",
            pad=12,
        )

        color = "#d62728"

        sample = get_sample_near_best_step(run)
        generated_text = get_generated_text_from_sample(sample)

        ax.text(
            0.00,
            0.98,
            run["name"],
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            color=color,
            va="top",
        )

        text = clean_sample_text(generated_text, max_chars=420)
        wrapped = textwrap.fill(text, width=44)

        ax.text(
            0.02,
            0.86,
            wrapped,
            transform=ax.transAxes,
            fontsize=18,
            color="black",
            va="top",
            linespacing=1.35,
        )


    fig, axes = plt.subplots(
        1,
        2,
        figsize=(24, 8),
        gridspec_kw={"width_ratios": [1.2, 1.05]},
    )

    plot_single_run_epoch_loss(axes[0], target_run)
    draw_single_run_sample_panel(axes[1], target_run)

    fig.suptitle(
        "basic_base_vocab2000_ctx64_emb128_layers4_epochs40 실험 비교",
        fontsize=20,
        fontweight="bold",
    )

    plt.tight_layout()
    plt.show()
else:
    print(f"[skip] {TARGET_RUN_NAME} 폴더를 찾지 못했습니다.")